In [3]:
import sys
import os


from tqdm import tqdm
import time


import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader


sys.path.append(os.path.abspath(".."))
from package.BPETokenizer import * 
from package.TextDataset import * 
from package.GPT import * 


%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
sliding_windows = 0.

context_window = 128
batch_size = 64

d_emb = 320
nb_heads = 4
d_k = d_emb // nb_heads

nb_layers = 5

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
#device = torch.device("cpu")
print(device)

mps


In [10]:
tokenizer = BPETokenizer()
tokens = torch.tensor(tokenizer.load_tokens("../tokens_bpe.tok"))
print(tokens.shape)


vocab_size = tokenizer.vocab_size
print("vocab_size : " , vocab_size)

dataset = TextDataset(tokens,context_window=context_window,sliding_windows=sliding_windows)
print("dataset_size : " ,len(dataset))


loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    num_workers = 2
)


Don't forget to load your tokenizer
torch.Size([3179243])
vocab_size :  4000
dataset_size :  3179116


In [11]:
gpt = GPT(vocab_size=vocab_size, context_window=context_window, d_emb=d_emb,nb_layers=nb_layers,nb_heads=nb_heads).to(device)


In [ ]:
nb_param = 0 
for name, param in gpt.named_parameters():
    c= 1
    for i in param.size():
        c *= i
    nb_param += c

    print(f"{name}: {param.device} {param.size()} {c}")

for name, buf in gpt.named_buffers():
    print(name, buf.device)
    
print(f"Total number of parameters : {nb_param}")



embedding.E.weight: mps:0 torch.Size([4000, 320]) 1280000
embedding.P.weight: mps:0 torch.Size([128, 320]) 40960
transformer_blocks.0.norm1.weight: mps:0 torch.Size([320]) 320
transformer_blocks.0.norm1.bias: mps:0 torch.Size([320]) 320
transformer_blocks.0.norm2.weight: mps:0 torch.Size([320]) 320
transformer_blocks.0.norm2.bias: mps:0 torch.Size([320]) 320
transformer_blocks.0.attention.Wq.weight: mps:0 torch.Size([320, 320]) 102400
transformer_blocks.0.attention.Wk.weight: mps:0 torch.Size([320, 320]) 102400
transformer_blocks.0.attention.Wv.weight: mps:0 torch.Size([320, 320]) 102400
transformer_blocks.0.attention.Wo.weight: mps:0 torch.Size([320, 320]) 102400
transformer_blocks.0.feed_forward.NN.0.weight: mps:0 torch.Size([960, 320]) 307200
transformer_blocks.0.feed_forward.NN.0.bias: mps:0 torch.Size([960]) 960
transformer_blocks.0.feed_forward.NN.2.weight: mps:0 torch.Size([320, 960]) 307200
transformer_blocks.0.feed_forward.NN.2.bias: mps:0 torch.Size([320]) 320
transformer_blo

In [13]:
print(len(loader))

49673


In [ ]:
counter = 0

saving_frequency = 1000

nb_steps = 10000

optimizer = torch.optim.Adam(gpt.parameters(), lr=3e-4)


#for step,(x, y) in enumerate((loader)):
for step in tqdm(range(nb_steps)):
    x, y = next(iter(loader))
    x = x.to(device)
    y = y.to(device)

    if step%saving_frequency==0:
        filename = f"training_historic/{counter:04d}.w"
        gpt.save(filename)
        counter+=1

    optimizer.zero_grad(set_to_none=True)
    logits = gpt(x)
    loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
    loss.backward()
    optimizer.step()

filename = f"training_historic/final.w"
gpt.save(filename)

100%|██████████| 10000/10000 [12:18:13<00:00,  4.43s/it]    


In [15]:
nb_steps = 20000
#for step,(x, y) in enumerate((loader)):
for step in tqdm(range(nb_steps)):
    x, y = next(iter(loader))
    x = x.to(device)
    y = y.to(device)

    if step%saving_frequency==0:
        filename = f"training_historic/{counter:04d}.w"
        gpt.save(filename)
        counter+=1

    optimizer.zero_grad(set_to_none=True)
    logits = gpt(x)
    loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
    loss.backward()
    optimizer.step()

filename = f"training_historic/final2.w"
gpt.save(filename)

100%|██████████| 20000/20000 [5:50:38<00:00,  1.05s/it]     
